# Weekly benchmarks

Parse the weekly ClimaCoupler and ClimaLand Buildkite jobs mirrored by the
`fetch-weekly-buildkite` stage and plot their scalar metrics over time.

The coupler runs are those reported into the CliMA Slack `#coupler-report`
channel: the target AMIP configuration (`climacoupler-amip`), the validation
suite (`climacoupler-longruns`), and the CPU/GPU comparison
(`climacoupler-cpu-gpu-benchmarks`). `climaland-long-runs` runs twice weekly,
once as a ~2 year run and once as a 19/20 year run.

Note that the land RMSE metrics behind the O3 OKRs are **not** available
here. ClimaLand computes them in its leaderboard extension and renders them
straight to PNG, so what these logs give for the land runs is throughput and
stability, not error against observations.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    print("Moving to repo root")
    os.chdir("..")

Moving to repo root


## Log parsing

Every metric below comes out of the job's raw log. Buildkite wraps output in
ANSI colour codes and OSC timestamp markers, so those are stripped first.

Note that ClimaCoupler's progress format changed around 2026-07-22. Older
logs print a block of `key = value` lines with `simulation_time` as a
human-readable duration (`"156 weeks, 6 days"`) alongside
`n_steps_completed` and `percent_complete`; newer ones print
`time = <date> (<elapsed>)` and `step = N (P%)`. Both are handled so the
series is continuous across the change.

In [2]:
import glob
import json
import math
import re
from datetime import datetime, timezone

import pandas as pd

ANSI = re.compile(r"\x1b\[[0-9;]*[a-zA-Z]|\x1b[_\]][^\x07]*\x07")
BK_TIMESTAMP = re.compile(r"^_bk;t=\d+", re.M)

# Units Julia prints in elapsed-time summaries like "1 y 354 d"
UNIT_DAYS = {"y": 365.0, "d": 1.0, "h": 1 / 24, "m": 1 / 1440, "s": 1 / 86400}


def clean_log(text: str) -> str:
    """Strip colour codes and Buildkite's per-line timestamp markers."""
    return BK_TIMESTAMP.sub("", ANSI.sub("", text))


def parse_elapsed_days(text: str) -> float | None:
    """'300 d' / '1 d 1 h' / '1 y 354 d' -> float days."""
    days = 0.0
    for value, unit in re.findall(r"([0-9.]+)\s*(y|d|h|m|s)\b", text):
        days += float(value) * UNIT_DAYS[unit]
    return days or None


# Units Julia's canonicalized durations use in the pre-2026-07-22 progress
# block, e.g. "156 weeks, 6 days". Months and years never appear, since
# neither is a fixed-length period.
CANONICAL_UNIT_DAYS = {
    "week": 7.0,
    "day": 1.0,
    "hour": 1 / 24,
    "minute": 1 / 1440,
    "second": 1 / 86400,
    "millisecond": 1 / 86_400_000,
    "microsecond": 1 / 86_400_000_000,
}


def parse_canonical_duration_days(text: str) -> float | None:
    """'156 weeks, 6 days' / '1 day, 10 hours' -> float days."""
    days = 0.0
    pattern = r"([0-9.]+)\s*(" + "|".join(CANONICAL_UNIT_DAYS) + r")s?\b"
    for value, unit in re.findall(pattern, text):
        days += float(value) * CANONICAL_UNIT_DAYS[unit]
    return days or None


def parse_duration_days(text: str | None) -> float | None:
    """Config durations like '732days' or '120secs' -> float days."""
    if not text:
        return None
    match = re.match(r"([0-9.]+)\s*(years?|days?|hours?|mins?|secs?)", text)
    if not match:
        return None
    return float(match.group(1)) * UNIT_DAYS[match.group(2)[0]]


def parse_run_metrics(raw_log: str) -> dict:
    """Extract the scalar metrics from one ClimaCoupler run log."""
    log = clean_log(raw_log)
    out: dict = {}

    # Throughput. ClimaCoupler runs print a final "Info: SYPD:" summary
    # line, which is the authoritative figure. ClimaLand runs never do --
    # they only report "estimated_sypd" inside each progress block -- so
    # fall back to the last finite one of those. Early blocks legitimately
    # read "Inf" before enough steps have elapsed to estimate, so those are
    # skipped rather than treated as a parse failure.
    sypd = re.findall(r"\bInfo: SYPD:\s*([0-9.eE+-]+)", log)
    if sypd:
        out["sypd"] = float(sypd[-1])
    else:
        out["sypd"] = None
        for value in re.findall(r'estimated_sypd = "([^"]*)"', log):
            try:
                number = float(value)
            except ValueError:
                continue
            if math.isfinite(number):
                out["sypd"] = number
    wps = re.findall(r"Walltime per coupling step:\s*([0-9.eE+-]+)", log)
    out["walltime_per_step_s"] = float(wps[-1]) if wps else None

    # How far the simulation got. For a crashed run this is the last
    # progress report before the exception, i.e. the run length before
    # the crash.
    progress = re.findall(
        r"Info: Progress\s*\n?\s*│?\s*time = (\S+) \(([^)]*)\)"
        r"\s*\n?\s*│?\s*step = (\d+) \(([0-9.]+)%\)",
        log,
    )
    if progress:
        sim_date, elapsed, step, percent = progress[-1]
        out["last_sim_date"] = sim_date
        out["last_sim_days"] = parse_elapsed_days(elapsed)
        out["last_step"] = int(step)
        out["percent_complete"] = float(percent)
    else:
        # Pre-2026-07-22 progress format
        out["last_sim_date"] = None
        sim_time = re.findall(r'simulation_time = "([^"]*)"', log)
        out["last_sim_days"] = (
            parse_canonical_duration_days(sim_time[-1]) if sim_time else None
        )
        steps = re.findall(r"n_steps_completed = (\d+)", log)
        out["last_step"] = int(steps[-1]) if steps else None
        percent = re.findall(r'percent_complete = "([0-9.]+)%"', log)
        out["percent_complete"] = float(percent[-1]) if percent else None

    # Config knobs. A run prints several config dicts (the coupler's, then
    # the atmos one it derives); only the coupler dict carries "job_id", so
    # key off that line to be sure t_end/dt/h_elem come from the same dict.
    config_line = ""
    for line in log.splitlines():
        if '"job_id" =>' in line and "Dict{" in line:
            config_line = line
            break
    match = re.search(r'"job_id" => "([^"]+)"', config_line)
    out["config_job_id"] = match.group(1) if match else None
    for key, cast in [
        ("t_end", str),
        ("dt", str),
        ("h_elem", int),
        ("z_elem", int),
        ("FLOAT_TYPE", str),
    ]:
        match = re.search(rf'"{key}" => "?([A-Za-z0-9_.]+)"?', config_line)
        out[key.lower()] = cast(match.group(1)) if match else None

    # Fraction of the configured run length actually simulated. This is the
    # crash-severity metric, and unlike raw simulated days it is comparable
    # across configs with different t_end. Prefer the percentage the model
    # reports itself; fall back to the ratio when a log predates it or the
    # progress block was cut off.
    out["t_end_days"] = parse_duration_days(out.get("t_end"))
    if out["percent_complete"] is not None:
        out["fraction_completed"] = out["percent_complete"] / 100.0
    elif out["last_sim_days"] and out["t_end_days"]:
        out["fraction_completed"] = out["last_sim_days"] / out["t_end_days"]
    else:
        out["fraction_completed"] = None

    # Crash signature: the first Julia-level error, which distinguishes
    # e.g. a fraction assertion from a NaN blowing up in a GPU kernel.
    match = re.search(r"^ERROR: (?:LoadError: )?(.{0,160})", log, re.M)
    out["error"] = match.group(1).strip() if match else None
    return out

## Build the table

In [3]:
EMOJI_TOKEN = re.compile(r":[a-z0-9_+-]+:")


def clean_job_name(name: str | None) -> str | None:
    """Strip Buildkite's ``:emoji:`` tokens from a job label."""
    if not name:
        return None
    return EMOJI_TOKEN.sub("", name).strip() or None


def parse_iso(text: str | None) -> datetime | None:
    if not text:
        return None
    dt = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


rows = []
for path in sorted(glob.glob("data/buildkite-weekly/*/*.json")):
    with open(path) as f:
        job = json.load(f)
    started = parse_iso(job.get("job_started_at") or job.get("job_created_at"))
    finished = parse_iso(job.get("job_finished_at"))
    row = {
        "date": started,
        "pipeline": job["pipeline"],
        "build_number": job["build_number"],
        "build_state": job["build_state"],
        "job_name": job["job_name"],
        "job_state": job["job_state"],
        "commit": job.get("commit"),
        "job_hours": (
            (finished - started).total_seconds() / 3600
            if started and finished
            else None
        ),
    }
    row.update(parse_run_metrics(job["raw_log_txt"]))
    # Identify the configuration. ClimaCoupler names it in its config dict;
    # ClimaLand has no equivalent, and its Buildkite job label is what
    # actually distinguishes the configs ("Soil" vs "Soil, 20 years"), so
    # fall back to that. It also rescues any coupler job whose config dict
    # did not parse.
    row["run_name"] = row["config_job_id"] or clean_job_name(row["job_name"])
    rows.append(row)

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

# A job that passed ran to completion by definition. Its last progress block
# is only the final checkpoint before the end -- the reporting interval
# differs per model (ClimaLand prints every 10%, the coupler every few
# percent) -- so the raw figure under-reports, and roughly half of all passed
# jobs would otherwise look like they stopped short. Pin them to 1.0 so this
# column means "how far did it get" rather than "when was progress last
# printed". Failed and cancelled jobs keep their parsed value, which is the
# number of interest for them.
df.loc[df["job_state"] == "passed", "fraction_completed"] = 1.0

print(f"{len(df)} jobs, {df['date'].min()} to {df['date'].max()}")
df["run_name"].value_counts()

1045 jobs, 2025-09-07 05:11:03.887000+00:00 to 2026-08-30 08:37:07.902000+00:00


run_name
amip                                                                51
cmip_edonly_land                                                    47
cmip_edonly_bucket                                                  47
gpu_amip_progedmf_1M_land_he16                                      40
gpu_amip_progedmf_1M_land_he30                                      40
                                                                    ..
GPU AMIP with prognostic EDMF + 1M + integrated land (30 helems)     1
GPU AMIP with diagnostic EDMF                                        1
amip_diagedmf_land_p2k_gpu                                           1
amip_diagedmf_land_gpu                                               1
GPU AMIP with diagnostic EDMF and IO                                 1
Name: count, Length: 65, dtype: int64

In [4]:
os.makedirs("results", exist_ok=True)
df.to_csv("results/weekly-benchmarks.csv", index=False)
df.tail(10)[
    ["date", "pipeline", "run_name", "job_state", "sypd", "fraction_completed"]
]

,date,pipeline,run_name,job_state,sypd,fraction_completed
1035,2026-08-30 05:16:04.099000+00:00,climacoupler-cpu-gpu-benchmarks,gpu_amip_progedmf,failed,NaN,NaN
1036,2026-08-30 08:37:04.926000+00:00,climacoupler-longruns,amip_progedmf_land,passed,1.857636,1.000
1037,2026-08-30 08:37:04.969000+00:00,climacoupler-longruns,cmip_edonly_land,failed,NaN,0.012
1038,2026-08-30 08:37:04.971000+00:00,climacoupler-longruns,amip_progedmf_1m_land,passed,1.246314,1.000
1039,2026-08-30 08:37:04.993000+00:00,climacoupler-longruns,amip_progedmf_land_p2k,passed,1.861584,1.000
1040,2026-08-30 08:37:05.026000+00:00,climacoupler-longruns,cmip_edonly_bucket,failed,NaN,0.012
1041,2026-08-30 08:37:05.028000+00:00,climacoupler-longruns,amip_edonly,passed,1.978792,1.000
1042,2026-08-30 08:37:05.073000+00:00,climacoupler-longruns,cmip_progedmf_land,failed,NaN,0.023
1043,2026-08-30 08:37:07.874000+00:00,climacoupler-longruns,slabplanet_aqua_evolve_ocean,passed,0.026263,1.000
1044,2026-08-30 08:37:07.902000+00:00,climacoupler-longruns,slabplanet_evolve_ocean,passed,0.026240,1.000


## Recent failures

What actually killed each run, alongside how far it got. The signature
distinguishes e.g. a surface-fraction assertion from a NaN blowing up
inside a GPU kernel.

In [5]:
# Job states that mean the run actually fell over. "canceled" is excluded
# throughout: a human stopping a run is not a model failure.
FAILURE_STATES = ["failed", "timed_out", "broken"]

failures = df[
    df["job_state"].isin(FAILURE_STATES) & df["error"].notna()
].sort_values("date", ascending=False)

failures.head(20)[
    ["date", "run_name", "job_state", "fraction_completed", "error"]
]

,date,run_name,job_state,fraction_completed,error
1042,2026-08-30 08:37:05.073000+00:00,cmip_progedmf_land,failed,0.023,a DomainError was thrown during kernel executi...
1040,2026-08-30 08:37:05.026000+00:00,cmip_edonly_bucket,failed,0.012,AssertionError: minimum((ice_fraction .+ land_...
1037,2026-08-30 08:37:04.969000+00:00,cmip_edonly_land,failed,0.012,AssertionError: minimum((ice_fraction .+ land_...
1035,2026-08-30 05:16:04.099000+00:00,gpu_amip_progedmf,failed,NaN,a DomainError was thrown during kernel executi...
1031,2026-08-30 05:16:04.029000+00:00,gpu_amip_progedmf_io,failed,NaN,a DomainError was thrown during kernel executi...
1023,2026-08-23 09:28:27.980000+00:00,cmip_progedmf_land,failed,0.006,AssertionError: minimum((ice_fraction .+ land_...
1018,2026-08-23 08:37:03.996000+00:00,cmip_edonly_bucket,failed,0.006,AssertionError: minimum((ice_fraction .+ land_...
1016,2026-08-23 08:37:03.940000+00:00,cmip_edonly_land,failed,0.001,AssertionError: minimum((ice_fraction .+ land_...
1001,2026-08-16 10:49:46.793000+00:00,cmip_progedmf_land,failed,0.200,a DomainError was thrown during kernel executi...
994,2026-08-16 08:37:03.991000+00:00,cmip_edonly_bucket,failed,0.047,a DomainError was thrown during kernel executi...


## Plot

Only the configurations present in each pipeline's most recent build are
plotted, so retired configs drop off on their own rather than needing a
hard-coded list here.

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configs still in use, i.e. those that ran in each pipeline's latest build
active = set()
for pipeline, group in df.dropna(subset=["run_name"]).groupby("pipeline"):
    latest = group[group["build_number"] == group["build_number"].max()]
    active.update(latest["run_name"])

plot_df = df[df["run_name"].isin(active)]
run_names = sorted(plot_df["run_name"].dropna().unique())
print(f"{len(run_names)} active configs: {', '.join(run_names)}")

20 active configs: GPU ClimaAtmos with prognostic EDMF, GPU ClimaAtmos without EDMF, Global bucket simulation, Snowy Land + P Model, Snowy Land + P Model (Optimal LAI), Soil, amip, amip_edonly, amip_progedmf_1m_land, amip_progedmf_land, amip_progedmf_land_p2k, cmip_edonly_bucket, cmip_edonly_land, cmip_progedmf_land, gpu_amip_progedmf, gpu_amip_progedmf_1M_land_he16, gpu_amip_progedmf_1M_land_he30, gpu_amip_progedmf_io, slabplanet_aqua_evolve_ocean, slabplanet_evolve_ocean


In [7]:
import plotly.colors as pc

# Dark24 has 24 distinct hues, enough that no two configs share a colour
PALETTE = pc.qualitative.Dark24

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Throughput (SYPD)",
        "Fraction of configured run length simulated",
        "Job wallclock (hours)",
    ),
)

for i, run_name in enumerate(run_names):
    sub = plot_df[plot_df["run_name"] == run_name].sort_values("date")
    color = PALETTE[i % len(PALETTE)]
    style = dict(
        legendgroup=run_name,
        mode="lines+markers",
        marker=dict(color=color),
        line=dict(color=color),
    )
    hover = sub[["job_state", "error", "pipeline"]].fillna("")

    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["sypd"],
            name=run_name,
            customdata=hover,
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>SYPD: %{y:.3f}<br>"
                "%{customdata[2]}<br>%{customdata[0]}<extra></extra>"
            ),
            **style,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["fraction_completed"] * 100,
            name=run_name,
            showlegend=False,
            customdata=sub[["job_state", "error", "last_sim_days"]].fillna(""),
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>%{y:.1f}% of target "
                "(%{customdata[2]:.1f} sim days)<br>"
                "%{customdata[0]}<br>%{customdata[1]}<extra></extra>"
            ),
            **style,
        ),
        row=2,
        col=1,
    )
    # Mark genuine failures so a crash reads as a crash rather than just a
    # low value (see FAILURE_STATES above)
    failed = sub[sub["job_state"].isin(FAILURE_STATES)]
    fig.add_trace(
        go.Scatter(
            x=failed["date"],
            y=failed["fraction_completed"] * 100,
            name=run_name,
            legendgroup=run_name,
            showlegend=False,
            mode="markers",
            marker=dict(color=color, symbol="x", size=11),
            hoverinfo="skip",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["job_hours"],
            name=run_name,
            showlegend=False,
            **style,
        ),
        row=3,
        col=1,
    )

# Throughput spans two orders of magnitude across these configs
# (slabplanet ~0.03 SYPD vs AMIP >2), so a linear axis hides all structure
fig.update_yaxes(title_text="SYPD", type="log", row=1, col=1)
fig.update_yaxes(title_text="% of target", range=[0, 105], row=2, col=1)
fig.update_yaxes(title_text="hours", row=3, col=1)
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_layout(
    height=900,
    title="Weekly ClimaCoupler and ClimaLand benchmarks",
    legend=dict(orientation="h", y=-0.09),
)

os.makedirs("figures", exist_ok=True)
fig.write_json("figures/weekly-benchmarks.json")
fig